In [ ]:
response = agent.go(
    "In a single pipeline: "
    "1) Download Bacillus subtilis 168 genome from NCBI using accession GCF_000009045.1 "
    "and E. coli K12 genome using accession GCF_000005845.2 — both in the same output folder. "
    "2) For each downloaded genome compute: total length, GC content, number of contigs, N50. "
    "3) Create a side-by-side bar chart with matplotlib comparing the two genomes "
    "and save it as genome_comparison.png. "
    "4) Save a summary table to comparison_report.txt."
)
print(response)



================================ Human Message =================================

In a single pipeline: 1) Download Bacillus subtilis 168 genome from NCBI using accession GCF_000009045.1 and E. coli K12 genome using accession GCF_000005845.2 — both in the same output folder. 2) For each downloaded genome compute: total length, GC content, number of contigs, N50. 3) Create a side-by-side bar chart with matplotlib comparing the two genomes and save it as genome_comparison.png. 4) Save a summary table to comparison_report.txt.
================================== Ai Message ==================================

Since this is a multi-step workflow that requires running software, code, or CLI tools, I will route this to the ORCHESTRATOR.

Here is the checklist:

- [ ] Step 1: Download Bacillus subtilis 168 genome from NCBI using accession GCF_000009045.1 and E. coli K12 genome using accession GCF_000005845.2, both in the same output folder, using `ncbi-genome-download`.
- [ ] Step 2: For each d

KeyboardInterrupt: 

## test ui 

In [ ]:
# ============================================================
# PREFLIGHT — Verifier Ollama est lance et llama3.1:8b present
# ============================================================
import requests as _req

OLLAMA_HOST = "http://localhost:11434"

# 1. Ping Ollama
try:
    _r = _req.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    _r.raise_for_status()
    _tags = _r.json()
    print(f"OK  Ollama repond sur {OLLAMA_HOST}")
except Exception as _e:
    print(f"ECHEC  Ollama ne repond pas : {_e}")
    print("  --> Demarrer Ollama avec : ollama serve")
    raise SystemExit("Ollama non disponible — arreter ici")

# 2. Verifier que llama3.1:8b est telecharge
_models = [m["name"] for m in _tags.get("models", [])]
_target = "llama3:8b"
_found  = any(_target in m for m in _models)
print(f"Modeles disponibles : {_models}")

if _found:
    print(f"OK  '{_target}' present")
else:
    print(f"ECHEC  '{_target}' introuvable")
    print(f"  --> Telecharger avec : ollama pull {_target}")
    raise SystemExit(f"Modele {_target} manquant — arreter ici")

print("\nPREFLIGHT PASSED — pret a continuer")


OK  Ollama repond sur http://localhost:11434
Modeles disponibles : ['llama3:latest', 'phi3:latest', 'llama3:8b']
OK  'llama3:8b' present

PREFLIGHT PASSED — pret a continuer


In [1]:
_ = agent.visualize_graph(mode="manual")


NameError: name 'agent' is not defined

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — Environnement & dépendances                    ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, os, subprocess, time

# Chemin racine du projet
UI_DIR    = r"C:\Users\PC\Downloads\agent genommer\agent\agent-ui"
AGENT_DIR = r"C:\Users\PC\Downloads\agent genommer\agent\genomeer\src"
sys.path.insert(0, AGENT_DIR)
os.chdir(UI_DIR)

# Variables d'environnement obligatoires
os.environ["AGENT_COPILOT_SECRET"]    = "test-local-secret-32chars-minimum!"
os.environ["ALLOWED_ORIGIN"]          = "http://localhost:8000"
os.environ["GENOMEER_SKIP_ENV_INSTALL"] = "1"
os.environ["TRANSFORMERS_OFFLINE"]    = "1"
os.environ["GENOMEER_RAG_OFFLINE"]    = "1"

# Installer les dépendances UI si besoin
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True, cwd=UI_DIR
)
if result.returncode != 0:
    print("PIP ERRORS:", result.stderr[-500:])
else:
    print("✅ Dépendances OK")


✅ Dépendances OK


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Démarrage du serveur FastAPI                   ║
# ╚══════════════════════════════════════════════════════════════╝
import threading, uvicorn, importlib

BASE_URL = "http://localhost:8000"

def _start_server():
    # Re-set env dans le thread uvicorn
    os.environ["AGENT_COPILOT_SECRET"] = "test-local-secret-32chars-minimum!"
    os.environ["ALLOWED_ORIGIN"]       = "http://localhost:8000"
    os.chdir(UI_DIR)
    sys.path.insert(0, UI_DIR)
    # Reload app modules proprement
    for mod in list(sys.modules.keys()):
        if mod.startswith("app"):
            del sys.modules[mod]
    uvicorn.run("app.main:app", host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

# Attendre que le serveur soit prêt
import requests as req
for i in range(20):
    try:
        r = req.get(f"{BASE_URL}/", timeout=2)
        print(f"✅ Serveur UP — status {r.status_code}  ({BASE_URL})")
        break
    except Exception:
        time.sleep(1)
        print(f"  Attente... ({i+1}/20)")
else:
    print("❌ Serveur non démarré après 20s — vérifier les erreurs ci-dessus")


c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


  Attente... (1/20)
  Attente... (2/20)
✅ Serveur UP — status 200  (http://localhost:8000)


In [ ]:
# ============================================================
# CELLULE 2 — Auth : register + login
# ============================================================
import requests as req, json

TEST_EMAIL    = "test@genomeer.local"
TEST_PASSWORD = "TestPass123!"
TEST_NAME     = "Test User"

# --- Register (ignore si deja existant) ---
r = req.post(f"{BASE_URL}/api/auth/register", json={
    "email": TEST_EMAIL, "name": TEST_NAME, "password": TEST_PASSWORD
})
if r.status_code == 200:
    print(f"OK Register  --> user_id={r.json()['id']}")
elif r.status_code == 400 and "already" in r.text:
    print("  Utilisateur deja existant -- OK")
else:
    print(f"ECHEC Register : {r.status_code} -- {r.text[:300]}")
    raise AssertionError(f"Register failed: {r.status_code}")

# --- Login ---
r = req.post(f"{BASE_URL}/api/auth/login", data={
    "username": TEST_EMAIL, "password": TEST_PASSWORD
})
assert r.status_code == 200, f"Login echoue : {r.status_code} -- {r.text[:300]}"
TOKEN = r.json()["access_token"]
AUTH  = {"Authorization": f"Bearer {TOKEN}"}
print(f"OK Login  --> token={TOKEN[:20]}...")

# --- Verifier /me ---
me = req.get(f"{BASE_URL}/api/auth/me", headers=AUTH)
assert me.status_code == 200, f"/me echoue : {me.text}"
print(f"OK /me  --> {me.json()}")


  Utilisateur deja existant -- OK
OK Login  --> token=eyJhbGciOiJIUzI1NiIs...
OK /me  --> {'id': 1, 'email': 'test@genomeer.local', 'name': 'Test User'}


In [ ]:
# ============================================================
# CELLULE 3 — Configurer le provider (Ollama local llama3:8b)
# ============================================================

OLLAMA_MODEL    = "llama3:8b"
OLLAMA_SOURCE   = "Ollama"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

print(f"Provider : {OLLAMA_SOURCE} @ {OLLAMA_BASE_URL}")
print(f"Modele   : {OLLAMA_MODEL}")

import json as _json
provider_payload = {
    "source":        OLLAMA_SOURCE,
    "base_url":      OLLAMA_BASE_URL,
    "api_key":       "",
    "default_model": OLLAMA_MODEL,   # champ obligatoire
}
r = req.post(f"{BASE_URL}/api/settings/provider", json=provider_payload, headers=AUTH)
if r.status_code == 200:
    print(f"OK  Provider configure : {OLLAMA_SOURCE} @ {OLLAMA_BASE_URL}")
else:
    print(f"ECHEC Provider config : {r.status_code} -- {r.text}")

model_name = OLLAMA_MODEL
r = req.post(f"{BASE_URL}/api/settings/models", json={
    "name":     model_name,
    "source":   OLLAMA_SOURCE,
    "base_url": OLLAMA_BASE_URL,
    "api_key":  "",
}, headers=AUTH)
if r.status_code == 200:
    print(f"OK  Modele ajoute : {model_name}")
elif "already" in r.text.lower() or r.status_code == 400:
    print(f"  Modele deja present : {model_name}")
else:
    print(f"ECHEC Modele : {r.status_code} -- {r.text}")

# user_models (pas 'models')
resp = req.get(f"{BASE_URL}/api/settings/models", headers=AUTH).json()
print(f"\nModeles en DB : {[m['name'] for m in resp.get('user_models', [])]}")
print(f"System default : {resp.get('system_default', {}).get('model')}")


Provider : Ollama @ http://localhost:11434/v1
Modele   : llama3:8b
OK  Provider configure : Ollama @ http://localhost:11434/v1
  Modele deja present : llama3:8b

Modeles en DB : ['llama3:8b']
System default : llama3:8b


In [ ]:
# ============================================================
# CELLULE 4 — Creer une session de chat
# ============================================================

model_name = "llama3:8b"   # repete ici pour eviter dependance sur CELLULE 3

r = req.post(f"{BASE_URL}/api/sessions", json={
    "model":            model_name,
    "interaction_mode": "auto",
}, headers=AUTH)
assert r.status_code == 200, f"Session create failed : {r.status_code} -- {r.text}"

session    = r.json()
SESSION_ID = session["id"]
print(f"OK Session creee  --> id={SESSION_ID}  modele={session.get('model')}")

sessions = req.get(f"{BASE_URL}/api/sessions", headers=AUTH).json()
print(f"Sessions actives : {[s['id'] for s in sessions]}")


OK Session creee  --> id=3  modele=llama3:8b
Sessions actives : [3, 1]


In [ ]:
# ============================================================
# CELLULE 5 — Test connexion : envoyer un message a l'agent
# ============================================================
import json

# Prompt simple adapte a llama3.1:8b (modele leger)
TEST_PROMPT = (
    "List the 3 most common bioinformatics file formats (FASTQ, FASTA, BAM). "
    "For each, write one sentence describing what it stores. "
    "Do NOT install anything. Do NOT run any code."
)

print(f"Envoi du message au session {SESSION_ID}...")
print(f"   Prompt : {TEST_PROMPT}\n")

with req.post(
    f"{BASE_URL}/api/sessions/{SESSION_ID}/messages",
    json={"message": TEST_PROMPT, "stream": True},
    headers=AUTH,
    stream=True,
    timeout=180,
) as resp:
    assert resp.status_code == 200, f"Chat failed : {resp.status_code} -- {resp.text[:300]}"

    events_received = []
    full_text       = []
    errors          = []

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        try:
            evt = json.loads(raw_line)
            events_received.append(evt.get("type", "?"))

            if evt.get("type") == "error":
                errors.append(evt.get("text", "unknown error"))
            elif evt.get("type") in ("message", "block"):
                text = evt.get("text", "")
                if text:
                    full_text.append(text)
            elif evt.get("type") == "done":
                print("OK  Stream termine (done recu)")
        except json.JSONDecodeError:
            pass

# Rapport
print(f"\n{'='*55}")
print(f"EVENEMENTS RECUS  : {events_received}")
print(f"ERREURS           : {errors if errors else 'aucune'}")
print(f"\nREPONSE AGENT :")
print("-"*55)
print("".join(full_text)[:1500] or "(aucun texte recu)")
print("-"*55)

# Assertions
assert "done" in events_received, "ECHEC  Aucun evenement 'done' -- stream incomplet"
assert not errors,                f"ECHEC  Erreurs agent : {errors}"
assert full_text,                 "ECHEC  Aucune reponse textuelle recue"
print("\nTEST CONNEXION UI <-> AGENT : PASSED")


Envoi du message au session 3...
   Prompt : List the 3 most common bioinformatics file formats (FASTQ, FASTA, BAM). For each, write one sentence describing what it stores. Do NOT install anything. Do NOT run any code.


BioAgent_v1 CONFIGURATION
DEFAULT CONFIG :
  Path: ./data
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: ollama/llama3.1
  Temperature: 0.7
  Use Tool Retriever: True

[AGENT LLM] Constructor Override:
  LLM Model: llama3:8b
  Source: Ollama
  Base URL: http://localhost:11434/v1

================================== Ai Message ==================================

<next:QA>

FASTQ: a text-based format that stores raw, unprocessed sequencing data, including the sequence, quality scores, and optional headers.

FASTA: a plain text format that stores a sequence of nucleotides or amino acids, often used for storing genomic or protein sequences.

BAM: a binary format that stores aligned sequencing data, including the mapped sequence, alignment coordinates, and additiona


BioAgent_v1 CONFIGURATION
DEFAULT CONFIG :
  Path: ./data
  Run Dir: .genomeer_runs
  Timeout Seconds: 600
  Llm: ollama/llama3.1
  Temperature: 0.7
  Use Tool Retriever: True

[AGENT LLM] Constructor Override:
  LLM Model: llama3:8b
  Source: Ollama
  Base URL: http://localhost:11434/v1

================================== Ai Message ==================================

A simple Q&A!
================================== Ai Message ==================================

A warm greeting! 

According to our recent history, I see you said "hi" earlier. In that case, I'll respond with a friendly "Hi back!"
================================== Ai Message ==================================

<next:QA>

Note: Since this question is a simple definition/explanation, I'm routing it to QA.
================================== Ai Message ==================================

I'm happy to help!

Based on your question, I'll define metagenomics in two sentences, including AI:

Metagenomics is the study of the ge

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Vérifier que l'historique est persisté en DB   ║
# ╚══════════════════════════════════════════════════════════════╝

r = req.get(f"{BASE_URL}/api/sessions/{SESSION_ID}/messages", headers=AUTH)
assert r.status_code == 200, f"History failed : {r.status_code}"

msgs = r.json()
print(f"Messages en DB : {len(msgs)}")
for m in msgs:
    role    = m.get("role", "?")
    content = (m.get("content") or "")[:120]
    logs_n  = len(m.get("logs", []))
    print(f"  [{role.upper():10s}] {content!r}  ({logs_n} log blocks)")

assert any(m["role"] == "user"      for m in msgs), "❌ Message utilisateur absent de la DB"
assert any(m["role"] == "assistant" for m in msgs), "❌ Réponse agent absente de la DB"
print("\n✅ PERSISTANCE DB : PASSED")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Test upload (path traversal bloqué + taille)   ║
# ╚══════════════════════════════════════════════════════════════╝
import io

# 7a — Upload normal
fake_fastq = b"@SEQ_1\nACGT\n+\nIIII\n@SEQ_2\nGGCC\n+\nHHHH\n"
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("test_sample.fastq", io.BytesIO(fake_fastq), "text/plain")},
    headers=AUTH)
assert r.status_code == 200, f"Upload échoué : {r.status_code} — {r.text}"
server_path = r.json()["path"]
print(f"✅ Upload OK → {server_path}")
assert "test_sample.fastq" in server_path, "Nom de fichier incorrect"

# 7b — Path traversal bloqué (C-1)
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("../../etc/passwd", io.BytesIO(b"malicious"), "text/plain")},
    headers=AUTH)
if r.status_code == 400:
    print(f"✅ Path traversal bloqué (400) : {r.json()['detail']}")
else:
    # Le basename() aura tronqué au nom seul, donc HTTP 200 avec nom "passwd"
    print(f"ℹ️  Path traversal neutralisé — fichier sauvé sous : {r.json().get('path', '?')}")
    assert "etc" not in r.json().get("path",""), "❌ Path traversal NON bloqué !"

# 7c — Fichier vide (cas limite)
r = req.post(f"{BASE_URL}/api/upload",
    files={"file": ("empty.fq", io.BytesIO(b""), "text/plain")},
    headers=AUTH)
print(f"ℹ️  Fichier vide → HTTP {r.status_code}")

print("\n✅ TEST UPLOAD : PASSED")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Résumé du test de connectivité UI ↔ Agent      ║
# ╚══════════════════════════════════════════════════════════════╝

checks = [
    ("Serveur FastAPI démarré",         True),
    ("Register / Login JWT",            True),
    ("Provider Ollama configuré",       True),
    ("Session créée",                   SESSION_ID is not None),
    ("Message envoyé + stream reçu",    "done" in events_received),
    ("Erreurs agent",                   not errors),
    ("Réponse textuelle non vide",      bool(full_text)),
    ("Historique persisté en DB",       True),
    ("Upload sécurisé",                 True),
]

print(f"\n{'='*55}")
print("  RAPPORT TEST CONNECTIVITÉ UI ↔ AGENT")
print(f"{'='*55}")
for label, ok in checks:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")

n_ok   = sum(1 for _, ok in checks if ok)
n_fail = len(checks) - n_ok
print(f"\n{'='*55}")
print(f"  {n_ok}/{len(checks)} checks passés" + ("  — TOUT OK" if n_fail == 0 else f"  — {n_fail} ÉCHEC(S)"))
print(f"{'='*55}")

if n_fail == 0:
    print("\n🎉 UI et agent sont bien connectés et fonctionnels.")
else:
    print("\n⚠️  Vérifier les erreurs ci-dessus.")
